# Multi-Agent Quorum: Parallel Claude Agents with Majority-Vote Enforcement

This notebook demonstrates a **quorum pattern** — three Claude agents that deliberate in parallel, each evaluating a decision from a different specialist angle, then voting. Enforcement only happens when at least two of three agents agree.

**Why quorum instead of a single agent?**

For high-stakes, irreversible actions (quarantining an account, blocking a user, triggering an alert), a single agent can be wrong. A quorum:
- Prevents false positives — one miscalibrated agent can't act unilaterally
- Provides explainability — three independent reasoning chains, not one
- Degrades safely — if one agent fails, it abstains rather than blocking the vote

**Real-world use case:** This pattern was built for [AI Sentinel Ecosystem](https://github.com/TanishkaMarrott/ai-sentinel-ecosystem), a production system that governs AWS cloud lab accounts — detecting abuse (GPU spin-ups, NAT Gateways, ECS clusters) and applying SCP quarantines only when 2/3 agents agree.

```
Abuse Signal
     │
     ├──▶ Safety Agent  (blast radius + exfiltration risk)  ──▶ APPROVE/REJECT/ABSTAIN
     ├──▶ Audit Agent   (CloudTrail evidence verification)  ──▶ APPROVE/REJECT/ABSTAIN  
     └──▶ Cost Agent    (financial exposure estimation)     ──▶ APPROVE/REJECT/ABSTAIN
                                    │
                            Tally votes (majority rules)
                                    │
                    ┌───────────────┼───────────────┐
                  2+ APPROVE     Mixed          2+ REJECT
                  ENFORCE        WARN           DISMISS
```

**What you'll build:**
1. Three specialist agents running in parallel via `ThreadPoolExecutor`
2. Structured JSON verdicts with confidence scores
3. A voting orchestrator that applies majority rules
4. Safe degradation — agent failures count as ABSTAIN, not errors

## Setup

In [ ]:
%pip install anthropic --quiet

In [ ]:
import json
import os
import concurrent.futures
from dataclasses import dataclass, field
from enum import Enum
from typing import Literal

import anthropic

# Set your API key or export ANTHROPIC_API_KEY in your environment
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()
MODEL = "claude-opus-4-6"

# DEMO_MODE=True returns hardcoded verdicts so the notebook runs without API calls
DEMO_MODE = os.getenv("DEMO_MODE", "false").lower() == "true"
print(f"DEMO_MODE: {DEMO_MODE}")

## 1. Data Models

Two simple dataclasses: the input signal and the per-agent verdict.

In [ ]:
class Verdict(str, Enum):
    APPROVE = "APPROVE"   # vote to enforce
    REJECT  = "REJECT"    # vote to dismiss
    ABSTAIN = "ABSTAIN"   # insufficient information / agent error


@dataclass
class AbuseSignal:
    account_id: str
    event_type: str        # e.g. CreateNatGateway
    lab_type: str          # e.g. AWS_S3 — defines what's in scope
    cost_estimate_usd: float
    cloudtrail_events: int # number of confirmed user-initiated events
    user_id: str
    region: str = "us-east-1"


@dataclass
class AgentVerdict:
    agent_role: str
    verdict: Verdict
    confidence: float      # 0.0 – 1.0
    reasoning: str
    recommended_action: str

## 2. The Three Specialist Agents

Each agent has a tightly scoped system prompt. They share the same agentic loop but reason about the signal from different angles.

**Key prompt design decisions:**
- `verdict` is constrained to exactly three values — no free-form output
- No markdown in the response — raw JSON only, so parsing is reliable
- Each agent knows its role but not what the others will say — independent reasoning

In [ ]:
VERDICT_FORMAT = """
Respond with ONLY raw JSON (no markdown, no code blocks):
{"verdict": "APPROVE", "confidence": 0.9, "reasoning": "one sentence", "recommended_action": "quarantine"}

verdict must be exactly one of: "APPROVE", "REJECT", "ABSTAIN"
  APPROVE = vote to enforce (the signal warrants action)
  REJECT  = vote to dismiss (insufficient evidence or low risk)
  ABSTAIN = cannot determine"""

SYSTEM_PROMPTS = {
    "safety": f"""You are the Safety Agent in a 3-agent quorum for high-stakes account governance.

Your role: assess OPERATIONAL RISK and blast radius.
- Is the resource type dangerous if left running? Could it enable data exfiltration or lateral movement?
- Is the resource clearly out-of-scope for the stated lab type?
- Do the CloudTrail events confirm this was user-initiated?
{VERDICT_FORMAT}""",

    "audit": f"""You are the Audit Agent in a 3-agent quorum for high-stakes account governance.

Your role: examine the EVIDENCE trail.
- Does the number of CloudTrail events indicate intentional action vs accidental single click?
- Does the pattern match known abuse (high-cost resource creation outside lab scope)?
- Is the evidence sufficient to justify enforcement?
{VERDICT_FORMAT}""",

    "cost": f"""You are the Cost Agent in a 3-agent quorum for high-stakes account governance.

Your role: assess FINANCIAL EXPOSURE.
- Estimate whether the cost justifies enforcement action.
- Thresholds: under $20/mo = warn only; $20-$100/mo = flag; over $100/mo = quarantine.
- APPROVE means the financial exposure warrants enforcement.
{VERDICT_FORMAT}""",
}

In [ ]:
def _parse_verdict(text: str, role: str) -> AgentVerdict:
    """Extract JSON verdict from agent response, with safe fallback."""
    try:
        start = text.find("{")
        end   = text.rfind("}") + 1
        if start >= 0 and end > start:
            data = json.loads(text[start:end])
            return AgentVerdict(
                agent_role=role,
                verdict=Verdict(data["verdict"]),
                confidence=float(data.get("confidence", 0.5)),
                reasoning=data.get("reasoning", "")[:300],
                recommended_action=data.get("recommended_action", "review"),
            )
    except (json.JSONDecodeError, ValueError, KeyError):
        pass

    # Fallback: infer verdict from text
    upper = text.upper()
    verdict = Verdict.APPROVE if "APPROVE" in upper else Verdict.REJECT if "REJECT" in upper else Verdict.ABSTAIN
    return AgentVerdict(role, verdict, 0.5, text[:200], "review")


def run_agent(role: str, signal: AbuseSignal) -> AgentVerdict:
    """Run one specialist agent and return its structured verdict."""
    if DEMO_MODE:
        # Hardcoded responses for demo — replace with real API call
        demo = {
            "safety": AgentVerdict("safety", Verdict.APPROVE, 0.91,
                "NAT Gateway is out of scope for S3-only lab and creates exfiltration risk.", "quarantine"),
            "audit":  AgentVerdict("audit",  Verdict.APPROVE, 0.95,
                "3 user-initiated events confirm intentional out-of-scope resource creation.", "quarantine"),
            "cost":   AgentVerdict("cost",   Verdict.APPROVE, 0.85,
                "$35/mo falls in the $20-$100 flag range, enforcement warranted.", "quarantine"),
        }
        return demo[role]

    user_message = (
        f"Evaluate this signal for account {signal.account_id}:\n"
        f"Event: {signal.event_type} | Lab type: {signal.lab_type}\n"
        f"Estimated cost: ${signal.cost_estimate_usd}/mo\n"
        f"CloudTrail events: {signal.cloudtrail_events} user-initiated\n"
        f"User: {signal.user_id} | Region: {signal.region}"
    )

    messages = [{"role": "user", "content": user_message}]
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        system=SYSTEM_PROMPTS[role],
        messages=messages,
    )
    text = "".join(b.text for b in response.content if hasattr(b, "text"))
    return _parse_verdict(text, role)

## 3. The Quorum Orchestrator

All three agents run in **parallel** via `ThreadPoolExecutor`. Quorum completes in the time of the slowest agent, not the sum. If an agent raises an exception, it automatically counts as ABSTAIN — the system degrades gracefully.

In [ ]:
def deliberate(signal: AbuseSignal) -> dict:
    """Run all three agents in parallel and tally their votes."""
    roles = ["safety", "audit", "cost"]

    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
        futures = {pool.submit(run_agent, role, signal): role for role in roles}
        verdicts = []
        for future in concurrent.futures.as_completed(futures):
            role = futures[future]
            try:
                verdicts.append(future.result())
            except Exception as e:
                # Agent failure → ABSTAIN, never blocks the vote
                verdicts.append(AgentVerdict(role, Verdict.ABSTAIN, 0.0, f"Error: {e}", "review"))

    votes_enforce = sum(1 for v in verdicts if v.verdict == Verdict.APPROVE)
    votes_dismiss = sum(1 for v in verdicts if v.verdict == Verdict.REJECT)

    if votes_enforce >= 2:
        decision, action = "ENFORCE", f"Quarantine SCP applied to account {signal.account_id}"
    elif votes_dismiss >= 2:
        decision, action = "DISMISS", "Signal closed — no action taken"
    else:
        decision, action = "WARN", f"Account {signal.account_id} flagged for human review"

    return {
        "signal": signal,
        "verdicts": sorted(verdicts, key=lambda v: v.agent_role),
        "votes_to_enforce": votes_enforce,
        "votes_to_dismiss": votes_dismiss,
        "final_decision": decision,
        "action": action,
    }

## 4. Run the Quorum

A NAT Gateway was created in an S3-only lab — classic abuse pattern.

In [ ]:
signal = AbuseSignal(
    account_id="123456789012",
    event_type="CreateNatGateway",
    lab_type="AWS_S3",          # S3-only lab — NAT Gateway is out of scope
    cost_estimate_usd=35.0,
    cloudtrail_events=3,
    user_id="kk_labs_user_123",
)

result = deliberate(signal)

print(f"Signal: {signal.event_type} | Account: {signal.account_id} | Lab: {signal.lab_type}")
print(f"Cost: ${signal.cost_estimate_usd}/mo | CloudTrail events: {signal.cloudtrail_events}\n")

print(f"{'Agent':<10} {'Verdict':<10} {'Conf':>6}  Reasoning")
print("-" * 85)
for v in result["verdicts"]:
    print(f"{v.agent_role:<10} {v.verdict.value:<10} {v.confidence:>5.0%}  {v.reasoning[:65]}")

print(f"\nFinal: {result['final_decision']} — {result['action']}")
print(f"Votes to enforce: {result['votes_to_enforce']} | Votes to dismiss: {result['votes_to_dismiss']}")

Expected output:

```
Signal: CreateNatGateway | Account: 123456789012 | Lab: AWS_S3
Cost: $35.0/mo | CloudTrail events: 3

Agent      Verdict      Conf  Reasoning
-------------------------------------------------------------------------------------
audit      APPROVE      95%  3 user-initiated events confirm intentional out-of-scope...
cost       APPROVE      85%  $35/mo falls in the $20-$100 flag range, enforcement...
safety     APPROVE      91%  NAT Gateway is out of scope for S3-only lab and creates...

Final: ENFORCE — Quarantine SCP applied to account 123456789012
Votes to enforce: 3 | Votes to dismiss: 0
```

## 5. Test the Edge Cases

The quorum degrades safely. Try signals that split the vote or fall below thresholds.

In [ ]:
# Low-cost signal — cost agent should REJECT, others may differ → WARN
low_cost_signal = AbuseSignal(
    account_id="999888777666",
    event_type="CreateSnapshot",
    lab_type="AWS_EC2",
    cost_estimate_usd=4.0,     # Under $20 threshold
    cloudtrail_events=1,       # Single event — could be accidental
    user_id="kk_labs_user_456",
)

result2 = deliberate(low_cost_signal)
print(f"Signal: {low_cost_signal.event_type} | Cost: ${low_cost_signal.cost_estimate_usd}/mo\n")
for v in result2["verdicts"]:
    print(f"{v.agent_role:<10} {v.verdict.value:<10} {v.confidence:>5.0%}  {v.reasoning[:65]}")
print(f"\nFinal: {result2['final_decision']} — {result2['action']}")

## Key Design Decisions

**1. Parallel over sequential**  
Three sequential agents would take 3× longer and share context — defeating the purpose of independent assessment. `ThreadPoolExecutor` gives true parallel execution; quorum time = slowest agent, not sum.

**2. ABSTAIN on failure, not error**  
If an agent crashes, it counts as ABSTAIN. A single failure can't trigger enforcement (no majority without it) and can't block a clear majority. The system never hard-fails.

**3. Constrained verdict schema**  
Agents return structured JSON with exactly three verdict values. No free-form output in the decision path — only the `reasoning` field is open text. This makes the voting logic deterministic and the output auditable.

**4. Specialist separation**  
Each agent knows its role but not the others' verdicts. Safety looks at blast radius. Audit looks at evidence. Cost looks at exposure. This prevents one agent's reasoning from anchoring the others.

## Adapting This Pattern

| Dimension | This example | Your use case |
|---|---|---|
| **Agents** | Safety, Audit, Cost | Legal, Technical, Business |
| **Input** | AWS abuse signal | Loan application, content moderation flag, deploy request |
| **Enforcement** | SCP quarantine | Block transaction, escalate ticket, pause deployment |
| **Threshold** | 2/3 majority | Configurable — raise to 3/3 for higher-stakes actions |

The pattern works for any decision where: the action is hard to reverse, a single model can be wrong, and independent specialist angles add signal.

**Production implementation:** [ai-sentinel-ecosystem](https://github.com/TanishkaMarrott/ai-sentinel-ecosystem) — the full system with AWS tool calls, SCP enforcement, and evaluation across 30 scenarios.